In [10]:
# =====================================================================
# PASSO 1: Importar as bibliotecas
# =====================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import requests

print("Bibliotecas importadas. Preparando o sistema meteorológico...")

# =====================================================================
# PASSO 2: Criar o Conhecimento Meteorológico (Dados Sintéticos)
# =====================================================================
# Como não temos um histórico real de décadas em mãos, vamos criar
# dados sintéticos ensinando a regra básica para a IA:
# Umidade alta (> 80%) e Pressão baixa (< 1010 hPa) = Chuva (1).
# Caso contrário = Sem chuva (0).

np.random.seed(42)
amostras = 1000

# Gerando dados aleatórios realistas
temperaturas = np.random.uniform(10, 40, amostras)       # 10 a 40 graus
umidades = np.random.uniform(30, 100, amostras)          # 30% a 100%
pressoes = np.random.uniform(990, 1030, amostras)        # 990 a 1030 hPa

# Juntando as 3 variáveis em uma matriz (1000 linhas, 3 colunas)
X_dados = np.column_stack((temperaturas, umidades, pressoes)).astype(np.float32)

# Regra para chover: Umidade > 75 e Pressão < 1012.
# O neurônio terá que descobrir essa relação sozinho!
y_dados = np.where((umidades > 75) & (pressoes < 1012), 1.0, 0.0).astype(np.float32)
y_dados = y_dados.reshape(-1, 1)

# Normalização (ESSENCIAL, como vimos no exemplo da idade)
X_mean = X_dados.mean(axis=0)
X_std = X_dados.std(axis=0)
X_norm = (X_dados - X_mean) / X_std

X_train = torch.from_numpy(X_norm)
y_train = torch.from_numpy(y_dados)

# =====================================================================
# PASSO 3: Criar e Treinar o Perceptron Classificador
# =====================================================================
# Agora temos 3 entradas (Temp, Umi, Pres) e 1 saída (Chance de chuva)
classificador_chuva = nn.Sequential(
    nn.Linear(3, 1),
    nn.Sigmoid() # Transforma a saída em uma probabilidade de 0 a 1
)

# Loss Function de Entropia Cruzada Binária (ideal para Sim/Não)
criterion = nn.BCELoss()
optimizer = optim.Adam(classificador_chuva.parameters(), lr=0.1)

print("\nTreinando a IA com padrões meteorológicos...")
for epoch in range(1000):
    optimizer.zero_grad()
    previsoes = classificador_chuva(X_train)
    loss = criterion(previsoes, y_train)
    loss.backward()
    optimizer.step()

print("IA Treinada com sucesso!")

# =====================================================================
# PASSO 4: Buscar Dados Reais na API e Prever
# =====================================================================
def prever_chuva_para_cidade(cidade):
    print(f"\n--- INICIANDO ANÁLISE PARA: {cidade.upper()} ---")
    try:
        # 4.1: Descobrir as coordenadas da cidade
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={cidade}&count=1&language=pt"
        geo_resp = requests.get(geo_url).json()

        if "results" not in geo_resp:
            print(f"Erro: Não consegui encontrar a cidade '{cidade}'.")
            return

        lat = geo_resp["results"][0]["latitude"]
        lon = geo_resp["results"][0]["longitude"]
        nome_oficial = geo_resp["results"][0]["name"]
        estado = geo_resp["results"][0].get("admin1", "Estado desconhecido")

        print(f"Localização encontrada: {nome_oficial} - {estado} (Lat: {lat:.2f}, Lon: {lon:.2f})")

        # 4.2: Buscar o clima atual
        clima_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,surface_pressure"
        clima_resp = requests.get(clima_url).json()

        temp_atual = clima_resp["current"]["temperature_2m"]
        umi_atual = clima_resp["current"]["relative_humidity_2m"]
        pres_atual = clima_resp["current"]["surface_pressure"]

        print("\n[Dados Meteorológicos Atuais da API]")
        print(f"* Temperatura: {temp_atual}°C")
        print(f"* Umidade:     {umi_atual}%")
        print(f"* Pressão:     {pres_atual} hPa")

        # 4.3: Preparar dados para a IA (Normalizar usando os mesmos parâmetros do treino)
        dados_reais = np.array([temp_atual, umi_atual, pres_atual], dtype=np.float32)
        dados_reais_norm = (dados_reais - X_mean) / X_std
        entrada_ia = torch.tensor([dados_reais_norm], dtype=torch.float32)

        # 4.4: Fazer a Previsão
        classificador_chuva.eval()
        with torch.no_grad():
            probabilidade = classificador_chuva(entrada_ia).item()

        # 4.5: Resultados
        chance_percentual = probabilidade * 100
        vai_chover = "SIM" if probabilidade > 0.5 else "NÃO"

        print("\n" + "="*40)
        print("VEREDITO DA INTELIGÊNCIA ARTIFICIAL")
        print("="*40)
        print(f"Probabilidade de Chuva: {chance_percentual:.1f}%")
        print(f"Previsão Final:         {vai_chover}")
        print("="*40)

    except Exception as e:
        print(f"Ocorreu um erro ao consultar a API: {e}")

# =====================================================================
# PASSO 5: Interação com o Usuário
# =====================================================================
cidade_alvo = input("Digite o nome da cidade para a previsão (Ex: Boituva): ")
prever_chuva_para_cidade(cidade_alvo)

Bibliotecas importadas. Preparando o sistema meteorológico...

Treinando a IA com padrões meteorológicos...
IA Treinada com sucesso!
Digite o nome da cidade para a previsão (Ex: Boituva): Londres

--- INICIANDO ANÁLISE PARA: LONDRES ---
Localização encontrada: Londres - Inglaterra (Lat: 51.51, Lon: -0.13)

[Dados Meteorológicos Atuais da API]
* Temperatura: 32.9°C
* Umidade:     16%
* Pressão:     1014.1 hPa

VEREDITO DA INTELIGÊNCIA ARTIFICIAL
Probabilidade de Chuva: 0.0%
Previsão Final:         NÃO
